# Emotion-vector dataset generation

This notebook is the reproducible run of the pilot pipeline. It builds a fully crossed set of stories: every premise is written once in each condition (`enraged`, `vulnerable`, `annoyed`, `happy`, `calm`, `neutral`), then a premise is kept only when all six stories pass.

The generation, filter, and judge implementations live in `pilot/`. This notebook is the procedure and the parameters. Later cells print the prompts they will use.

## Google Colab

1. Upload this project folder to Google Drive. It must contain `pilot/` and `requirements.txt`. The default location is `My Drive/SPAR Plan B`. If you use another folder, set `DRIVE_PROJECT` in the setup cell.
2. Open this notebook in Colab. Use a CPU runtime.
3. In the secrets panel (the key icon), add `OPENROUTER_API_KEY`. Add `HF_TOKEN` only if you set `PUSH_TO_HUB = True`.
4. Run all cells. The first run asks for permission to mount Drive, then installs the Python packages.
5. Checkpoints are written under the project `data/` folder on Drive. After a disconnect, run all cells again with the same `TAG` and the run resumes.

Locally, run it from the project folder. Keys come from the environment or from `.env`.


## Reproducibility

- `SEED` fixes character draws, diversity axes, and the train/test split. The split is by premise: every story from one premise stays in the same split.
- `TAG` names the checkpoint folder. The same tag resumes the same premises and stories, including after a Colab runtime restart, as long as the checkpoints are on Drive.
- Story temperature is 1.0. Deleting a checkpoint and generating again will not reproduce the same sentences. The saved parquet is the scenario set; the seed is what makes the split and the assignments repeatable.
- Keys are not written into this notebook. Colab reads them from the secrets panel. A local run reads `OPENROUTER_API_KEY` and, if you upload, `HF_TOKEN` from the environment or `.env`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Folder on Google Drive that contains pilot/. Ignored on a local run.
DRIVE_PROJECT = "/content/drive/MyDrive/SPAR Plan B"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_project() -> Path | None:
    candidates = [Path.cwd().resolve(), *list(Path.cwd().resolve().parents)[:6]]
    if IN_COLAB:
        candidates.append(Path(DRIVE_PROJECT))
        for base in (Path("/content"), Path("/content/drive/MyDrive")):
            if base.exists():
                candidates.extend(path for path in base.iterdir() if path.is_dir())
    for path in candidates:
        if (path / "pilot" / "__init__.py").exists():
            return path.resolve()
    return None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = find_project()
if ROOT is None:
    raise SystemExit(
        "Could not find pilot/. On Colab, upload the project folder to Drive "
        "and set DRIVE_PROJECT to that folder, then rerun this cell."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# A copy that lives only on the Colab disk disappears when the runtime stops.
# Keep checkpoints on Drive in that case. A project that is already on Drive
# writes data/ next to pilot/ and needs no override.
if IN_COLAB and not str(ROOT).startswith("/content/drive"):
    os.environ["PILOT_DATA_DIR"] = "/content/drive/MyDrive/emotion-vector-pilot/data"

if IN_COLAB:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")]
    )

if IN_COLAB:
    from google.colab import userdata
    for key in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        if os.environ.get(key, "").strip():
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = ""
        if value:
            os.environ[key] = value

print("project", ROOT)
print("checkpoints", os.environ.get("PILOT_DATA_DIR") or str(ROOT / "data"))
print("colab" if IN_COLAB else "local")


In [ ]:
import importlib.util

required = [
    "openai", "pandas", "numpy", "sklearn", "pyarrow",
    "faker", "sentence_transformers", "huggingface_hub",
]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise SystemExit(
        "Missing packages: " + ", ".join(missing) + ". "
        "On Colab, use Runtime > Restart session, then run all cells again. "
        "Locally, run: python -m pip install -r requirements.txt"
    )
print("packages ok")


## Run parameters

`TAG = "full"` resumes the pilot that is already on disk. Use a new tag to start a separate sample. `N_PREMISES` is the pool generated before filtering; `KEEP` is how many complete premises are written into the final parquet.


In [ ]:
SEED = 20261004
TAG = "full"
N_PREMISES = 130
KEEP = 100
PUSH_TO_HUB = False  # private Hugging Face upload; requires HF_TOKEN

assert N_PREMISES >= KEEP
print(f"seed={SEED}  tag={TAG}  premise pool={N_PREMISES}  keep={KEEP}")


In [ ]:
from pilot.config import (
    CONDITION_NAMES,
    CONDITIONS,
    DOC_MODEL,
    GLOSS,
    IDEA_MODEL,
    JUDGE_MODEL,
    PREMISE_DEDUP_COSINE,
    PROMPT_VERSION,
    RANK_RATIONALE,
    STORY_TEMPERATURE,
)
from pilot.llm import load_env
import os

load_env()
if not os.environ.get("OPENROUTER_API_KEY", "").strip():
    raise SystemExit("Set OPENROUTER_API_KEY. On Colab, add it in the secrets panel. Locally, export it or put it in .env.")

print("prompt_version", PROMPT_VERSION)
print("premise/judge model", IDEA_MODEL)
print("story model", DOC_MODEL, "temperature", STORY_TEMPERATURE)
print("judge model", JUDGE_MODEL)
print("premise dedup cosine", PREMISE_DEDUP_COSINE)
print(RANK_RATIONALE)
print("hub token set" if os.environ.get("HF_TOKEN", "").strip() else "hub token not set")


## Specification that will be applied

These strings are the ones `pilot/` will send. Premise generation also assigns a life domain, a social context, an event type, and a Faker name, then drops near-duplicates and premises that cannot plausibly take all six conditions.


In [ ]:
import pandas as pd
from pilot.premises import PREMISE_PROMPT
from pilot.stories import STORY_SYSTEM

conditions = pd.DataFrame(CONDITIONS)
print(conditions.to_string(index=False))
print("\n--- glosses ---")
for name in CONDITION_NAMES:
    if name == "neutral":
        print(f"{name}: matter-of-fact account, no emotional reaction")
    else:
        print(f"{name}: {GLOSS[name]}")
print("\n--- premise prompt ---\n")
print(PREMISE_PROMPT)
print("--- story system ---\n")
print(STORY_SYSTEM)


## Offline checks

Leakage patterns, word count, and the 80/20 premise split are checked here without calling a model.


In [ ]:
from pilot.selftest import run

run()


## 1. Premises

Generates `N_PREMISES` emotionally open premises. A rerun with the same `TAG` loads `data/<TAG>/ckpt_premises.jsonl` and only fills a shortfall.


In [ ]:
from pilot.premises import generate_premises

premises = generate_premises(N_PREMISES, TAG, seed=SEED)
premises.head(3)


In [ ]:
from pilot.embeddings import embed, max_pairwise, nearest_neighbor
import numpy as np

print(f"{len(premises)} premises")
print(premises.groupby(["life_domain", "social_context"]).size().unstack(fill_value=0))
vectors = embed(premises["premise"].tolist())
nearest = nearest_neighbor(vectors)
print(
    f"max pairwise cosine {max_pairwise(vectors):.3f}  "
    f"nearest-neighbor p50 {float(np.percentile(nearest, 50)):.3f}"
)
premises[["premise_id", "character_name", "life_domain", "social_context", "event_type", "premise"]]


## 2–4. Stories, filters, and the blind judge

Each premise is written in all six conditions. Stories resume from `ckpt_stories.jsonl`. Filters then drop generation errors, lengths outside 150–250 words, emotion-word leakage, and premise contradictions. The judge never sees the target label. A story is kept when the label matches and the intensity is at least 3, or at most 1 for neutral.

If fewer than `KEEP` premises pass in every condition, the pool is topped up and only the new rows are generated.


In [ ]:
from pilot.export import complete_premises
from pilot.stories import generate_stories
from pilot.validate import validate_stories

pool = N_PREMISES
validated = None
for round_index in range(4):
    raw = generate_stories(premises, TAG, seed=SEED)
    validated = validate_stories(raw, TAG)
    ready = complete_premises(validated)
    print(f"complete premises: {len(ready)} (need {KEEP})", flush=True)
    if len(ready) >= KEEP or round_index == 3:
        break
    raw_n = validated["premise_id"].nunique()
    yield_rate = (len(ready) / raw_n) if raw_n else 0.5
    pool += int((KEEP - len(ready)) / max(yield_rate, 0.4)) + 5
    print(f"topping up the premise pool to {pool}", flush=True)
    premises = generate_premises(pool, TAG, seed=SEED)

print(f"passed {int(validated['passed_all'].fillna(False).sum())} / {len(validated)}")
pd.crosstab(validated["condition"], validated["judge_label"].fillna("(none)"), dropna=False)


In [ ]:
print(validated.groupby("condition")["passed_all"].mean().round(3).to_string())
print()
print(validated.groupby("condition")["word_count"].agg(["mean", "min", "max"]).round(1).to_string())


## 5–6. Split and export

The first `KEEP` premises that passed in all six conditions are kept, in premise-id order. The test split is 20 premises when at least 100 are kept, otherwise 20% of the premises. Stories from one premise are never split across train and test.


In [ ]:
import json
from pilot.checks import run_checks
from pilot.config import PROMPT_VERSION, RANK_RATIONALE
from pilot.export import export_hub, export_local, select_final, write_spotcheck
from pilot.premises import data_dir

final = select_final(validated, keep=KEEP, seed=SEED)
print(
    f"kept {final['premise_id'].nunique()} premises, {len(final)} stories, "
    f"test premises {(final.drop_duplicates('premise_id')['split'] == 'test').sum()}"
)
exported = export_local(final, TAG)
write_spotcheck(final, TAG, seed=SEED)
checks = run_checks(final, TAG)
if PUSH_TO_HUB:
    export_hub(exported, path=str(data_dir(TAG) / "pilot.parquet"))

manifest = {
    "seed": SEED,
    "tag": TAG,
    "prompt_version": PROMPT_VERSION,
    "rank_rationale": RANK_RATIONALE,
    "n_stories": int(len(exported)),
    "n_premises": int(exported["premise_id"].nunique()) if len(exported) else 0,
    "n_test_premises": int((exported.drop_duplicates("premise_id")["split"] == "test").sum()) if len(exported) else 0,
    "premise_pool": int(premises["premise_id"].nunique()) if len(premises) else 0,
}
(data_dir(TAG) / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("manifest", manifest)
final.groupby(["split", "condition"]).size().unstack(fill_value=0)


## Confound readout

A length-only classifier should sit near chance (one in six). A TF-IDF classifier is expected to score high; its top features are the place to look for leaked emotion words or a repeated gesture. Within-condition nearest-neighbor cosine flags a condition that collapsed onto one template.


In [ ]:
length = checks["length_table"]
print(
    "length means within 10%:",
    length["within_10_percent"],
    " span",
    length["max_mean_over_min_mean_minus_1"],
)
print("length classifier", checks["length_classifier_accuracy"])
print("tfidf classifier", checks["tfidf_classifier_accuracy"])
print("within-condition nearest neighbor")
for condition, stats in checks["within_condition_diversity"].items():
    print(f"  {condition:12} p50 {stats['nn_p50']:.3f}  p90 {stats['nn_p90']:.3f}  max {stats['nn_max']:.3f}")
print("\ntop TF-IDF features")
for condition, features in checks["tfidf_top_features"].items():
    terms = ", ".join(item["feature"] for item in features[:5])
    print(f"  {condition:12} {terms}")


## Outputs

| File | What it is |
|---|---|
| `data/<TAG>/pilot.parquet` | Kept stories, with the train/test split |
| `data/<TAG>/premises.parquet` | Premise pool, including any that were topped up |
| `data/<TAG>/stories.parquet` | Every generated story and its judge result |
| `data/<TAG>/checks.json` | Length, TF-IDF, and diversity checks |
| `data/<TAG>/spotcheck_blind.csv` | 10 stories per condition, labels hidden, for two human raters |
| `data/<TAG>/spotcheck_key.csv` | The target condition for those 60 stories |
| `data/<TAG>/manifest.json` | Seed, prompt version, and split sizes |

Human agreement (Cohen's kappa) is not computed here. It needs the two rater columns in the blind file filled in.
